In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

link_df = "https://raw.githubusercontent.com/Ptit-Lulu/Projet2_Cinema_Creuse/donnees_df/data_frame_finale2.csv" 

df = pd.read_csv(link_df,sep=';')

In [2]:
df

,tconst,title,startYear,runtimeMinutes,spoken_languages,primaryName,primaryProfession,knownForTitles,averageRating,popularity,tagline,overview,poster_path,video,budget,revenue,genre_1,genre_2,genre_3
0,tt0035423,Kate et Léopold,2001,118,"['en', 'fr', 'it']",James Mangold,"producer,director,writer","tt3315342,tt11563598,tt1950186,tt0358273",6.4,15.770,"If they lived in the same century, they'd be p...",When her scientist ex-boyfriend discovers a po...,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg,False,48000000,76019048,Comedy,Fantasy,Romance
1,tt0052614,Le bel âge,1960,100,['fr'],Pierre Kast,"director,writer,assistant_director","tt0052614,tt0057635,tt0208040,tt0158774",6.8,1.400,Indisponible,"Steph, Jean-Claude and Jacques work in a Paris...",/98Z5yPoNIm8sQeyep5cpr5NHov9.jpg,False,0,0,Comedy,Drama,NaN
2,tt0052686,Certains l'aiment... froide!,1960,90,['fr'],Jean Bastia,"assistant_director,director,writer","tt0315099,tt0054394,tt0052686,tt0159605",5.2,1.916,Indisponible,The old Valmorin died 200 years ago. The notar...,/nWxNWddjRO14xGOiHU770oDfXsV.jpg,False,0,0,Comedy,NaN,NaN
3,tt0052698,Classe tous risques,1960,103,"['fr', 'it']",Claude Sautet,"writer,director,assistant_director","tt0105682,tt0113947,tt0064165,tt0075975",7.5,5.843,Tough Gangster Action,"On crowded Milan streets, two men execute a sp...",/uMiuUAlPwiWDKSgu4SoShpzvMR.jpg,False,0,0,Crime,Drama,Romance
4,tt0052737,Le dialogue des Carmélites,1960,112,['fr'],Inconnu,Inconnu,Inconnu,7.0,2.701,Indisponible,This drama about the Carmelite order of nuns i...,/5yW8duwP8vT0ucs8HtqtLY5neFW.jpg,False,0,0,Drama,History,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7980,tt9845110,Deux,2019,99,"['de', 'fr']",Filippo Meneghetti,"writer,director,assistant_director","tt9845110,tt3454720,tt2447032,tt1054112",7.2,10.742,Indisponible,Nina and Madeleine have been sharing their liv...,/itPHi7V84mUOYuvD3t0QK2oWfVP.jpg,False,0,208723,Drama,Romance,NaN
7981,tt9850344,Police,2020,98,['fr'],Anne Fontaine,"actress,writer,director","tt4370784,tt1035736,tt5989394,tt0119773",5.4,10.722,Indisponible,Three officers are tasked with escorting an il...,/52KQaRQiEIZ9TE4P5iIrRIK7Wej.jpg,False,0,0,Crime,Drama,Thriller
7982,tt9861204,The Love Europe Project,2019,99,"['hr', 'cs', 'en', 'fr', 'de', 'it', 'no', 'pl...",Inconnu,Inconnu,Inconnu,6.7,1.400,Indisponible,The Love Europe Project is a collection of 9 s...,/7TzHFu7InuyhJ2xUiHfJ07aszTm.jpg,False,0,0,Drama,NaN,NaN
7983,tt9894450,Felicità,2020,81,['fr'],Bruno Merle,"writer,producer,director","tt14485734,tt9894450,tt23898914,tt0814143",6.6,5.208,Indisponible,"Tommy, 11 years old, is on the road again with...",/yhAiRXQUkMRnalRZFr1jiRCohjM.jpg,False,0,236880,Comedy,Drama,NaN


In [3]:
df["overview"]

0       When her scientist ex-boyfriend discovers a po...
1       Steph, Jean-Claude and Jacques work in a Paris...
2       The old Valmorin died 200 years ago. The notar...
3       On crowded Milan streets, two men execute a sp...
4       This drama about the Carmelite order of nuns i...
                              ...                        
7980    Nina and Madeleine have been sharing their liv...
7981    Three officers are tasked with escorting an il...
7982    The Love Europe Project is a collection of 9 s...
7983    Tommy, 11 years old, is on the road again with...
7984    A psychiatric hospital patient pretends to be ...
Name: overview, Length: 7985, dtype: str

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7985 entries, 0 to 7984
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             7985 non-null   str    
 1   title              7985 non-null   str    
 2   startYear          7985 non-null   int64  
 3   runtimeMinutes     7985 non-null   int64  
 4   spoken_languages   7985 non-null   str    
 5   primaryName        7985 non-null   str    
 6   primaryProfession  7985 non-null   str    
 7   knownForTitles     7985 non-null   str    
 8   averageRating      7985 non-null   float64
 9   popularity         7985 non-null   float64
 10  tagline            7985 non-null   str    
 11  overview           7985 non-null   str    
 12  poster_path        7985 non-null   str    
 13  video              7985 non-null   bool   
 14  budget             7985 non-null   int64  
 15  revenue            7985 non-null   int64  
 16  genre_1            7985 non-null   

In [5]:
import nltk
import os

venv_nltk_path = r"J:\Projet GIT\Projet2_Cinema_Creuse\.venv\nltk_data"
nltk.data.path.insert(0, venv_nltk_path)
os.environ["NLTK_DATA"] = venv_nltk_path

# Télécharger les stopwords dans ce dossier
nltk.download('stopwords', download_dir=venv_nltk_path)

[nltk_data] Downloading package stopwords to J:\Projet
[nltk_data]     GIT\Projet2_Cinema_Creuse\.venv\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
import re
import nltk
from textblob import TextBlob as tb

def clean_fr(phrase):
    # Nettoyer ponctuation
    phrase = re.sub(r"[.,'’`;:!?]+", ' ', phrase)
    
    # Analyse TextBlob
    blob = tb(phrase.lower())
    
    # Lemmatisation
    tokens = [w.lemmatize() for w in blob.words]
    
    # Filtrer stopwords français
    stop_words = set(nltk.corpus.stopwords.words("french"))
    tokens = [t for t in tokens if t not in stop_words]
    
    return " ".join(tokens)

def clean_en(phrase):
    # Nettoyer ponctuation
    phrase = re.sub(r"[.,'’`;:!?]+", ' ', phrase)
    
    # Analyse TextBlob
    blob = tb(phrase.lower())
    
    # Lemmatisation
    tokens = [w.lemmatize() for w in blob.words]
    
    # Filtrer stopwords français
    stop_words = set(nltk.corpus.stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop_words]
    
    return " ".join(tokens)

In [ ]:
X = df[["title","startYear", "runtimeMinutes","primaryName","tagline", "overview", "budget", "revenue" ,"genre_1", "genre_2", "genre_3"]]

for col in ['title', 'tagline', 'overview']:
    X[col] = X[col].replace("Inconnu", "")

X['title'] = X['title'].apply(clean_fr)
X['tagline'] = X['tagline'].apply(clean_en)
X['overview'] = X['overview'].apply(clean_en)

nfeature = ["startYear", "runtimeMinutes", "budget", "revenue"]
cfeature = ["primaryName","genre_1", "genre_2", "genre_3"]
tfeature = ["title","tagline", "overview"]

preprocessing = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), nfeature),
        ('cat', OneHotEncoder(handle_unknown="ignore"), cfeature),
        ('text_title', TfidfVectorizer(), 'title'),
        ('text_tagline', TfidfVectorizer(), 'tagline'),
        ('text_overview', TfidfVectorizer(), 'overview')
    ])
        

In [8]:
X

,title,startYear,runtimeMinutes,primaryName,tagline,overview,budget,revenue,genre_1,genre_2,genre_3
0,when her scientist ex-boyfriend discovers a po...,2001,118,James Mangold,scientist ex-boyfriend discovers portal travel...,scientist ex-boyfriend discovers portal travel...,48000000,76019048,Comedy,Fantasy,Romance
1,steph jean-claude and jacques work in a parisi...,1960,100,Pierre Kast,steph jean-claude jacques work parisian art sh...,steph jean-claude jacques work parisian art sh...,0,0,Comedy,Drama,NaN
2,the old valmorin died 200 year ago the notary ...,1960,90,Jean Bastia,old valmorin died 200 year ago notary tell fam...,old valmorin died 200 year ago notary tell fam...,0,0,Comedy,NaN,NaN
3,crowded milan street two men execute a split-s...,1960,103,Claude Sautet,crowded milan street two men execute split-sec...,crowded milan street two men execute split-sec...,0,0,Crime,Drama,Romance
4,this drama about the carmelite order of nun is...,1960,112,Inconnu,drama carmelite order nun set french revolutio...,drama carmelite order nun set french revolutio...,0,0,Drama,History,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7980,nina and madeleine have been sharing their lif...,2019,99,Filippo Meneghetti,nina madeleine sharing life landing two apartm...,nina madeleine sharing life landing two apartm...,0,208723,Drama,Romance,NaN
7981,three officer are tasked with escorting an ill...,2020,98,Anne Fontaine,three officer tasked escorting illegal immigra...,three officer tasked escorting illegal immigra...,0,0,Crime,Drama,Thriller
7982,the love europe project is a collection of 9 s...,2019,99,Inconnu,love europe project collection 9 short film va...,love europe project collection 9 short film va...,0,0,Drama,NaN,NaN
7983,tommy 11 year old is the road again with her e...,2020,81,Bruno Merle,tommy 11 year old road eccentric parent time f...,tommy 11 year old road eccentric parent time f...,0,236880,Comedy,Drama,NaN
